In [ ]:
# LSTM for Sequence Labeling (POS Tagging on UD English-EWT)
# This notebook cell covers:
# 1) loading CoNLL-U data, 2) preprocessing, 3) BiLSTM model, 4) training, 5) evaluation

from pathlib import Path
from collections import Counter, defaultdict
import random
import math

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


DATA_DIR = Path("UD_English-EWT")
TRAIN_PATH = DATA_DIR / "en_ewt-ud-train.conllu"
DEV_PATH = DATA_DIR / "en_ewt-ud-dev.conllu"
TEST_PATH = DATA_DIR / "en_ewt-ud-test.conllu"

for p in [TRAIN_PATH, DEV_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Could not find dataset file: {p}")

def read_conllu(path: Path):
    sentences = []
    tags = []

    cur_words, cur_tags = [], []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                if cur_words:
                    sentences.append(cur_words)
                    tags.append(cur_tags)
                cur_words, cur_tags = [], []
                continue

            if line.startswith("#"):
                continue

            parts = line.split("\t")
            if len(parts) < 4:
                continue

            token_id = parts[0]
            # Skip multiword token lines (e.g., 1-2) and empty nodes (e.g., 5.1)
            if "-" in token_id or "." in token_id:
                continue

            word = parts[1]
            upos = parts[3]

            cur_words.append(word)
            cur_tags.append(upos)

    if cur_words:
        sentences.append(cur_words)
        tags.append(cur_tags)

    return sentences, tags

train_sents, train_tags = read_conllu(TRAIN_PATH)
dev_sents, dev_tags = read_conllu(DEV_PATH)
test_sents, test_tags = read_conllu(TEST_PATH)

print(f"Train sentences: {len(train_sents)}")
print(f"Dev sentences:   {len(dev_sents)}")
print(f"Test sentences:  {len(test_sents)}")


PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_TAG = "<PAD_TAG>"

word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}

tag_set = set()
for sent in train_sents:
    for w in sent:
        wl = w.lower()
        if wl not in word2idx:
            word2idx[wl] = len(word2idx)

for seq in train_tags:
    tag_set.update(seq)

tag_list = sorted(tag_set)
tag2idx = {PAD_TAG: 0}
for t in tag_list:
    tag2idx[t] = len(tag2idx)

idx2tag = {i: t for t, i in tag2idx.items()}

print(f"Vocab size: {len(word2idx)}")
print(f"Number of POS tags: {len(tag2idx) - 1}")
print(f"Tags: {tag_list}")


class SequenceLabelingDataset(Dataset):
    def __init__(self, sentences, tags, word2idx_map, tag2idx_map):
        self.sentences = sentences
        self.tags = tags
        self.word2idx_map = word2idx_map
        self.tag2idx_map = tag2idx_map

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        labels = self.tags[idx]

        x = [self.word2idx_map.get(w.lower(), self.word2idx_map[UNK_TOKEN]) for w in words]
        y = [self.tag2idx_map[t] for t in labels]

        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


def collate_batch(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs], dtype=torch.long)

    max_len = int(lengths.max().item())
    x_padded = torch.full((len(xs), max_len), word2idx[PAD_TOKEN], dtype=torch.long)
    y_padded = torch.full((len(ys), max_len), tag2idx[PAD_TAG], dtype=torch.long)

    for i, (x, y) in enumerate(zip(xs, ys)):
        x_padded[i, : len(x)] = x
        y_padded[i, : len(y)] = y

    return x_padded, y_padded, lengths

BATCH_SIZE = 32

train_dataset = SequenceLabelingDataset(train_sents, train_tags, word2idx, tag2idx)
dev_dataset = SequenceLabelingDataset(dev_sents, dev_tags, word2idx, tag2idx)
test_dataset = SequenceLabelingDataset(test_sents, test_tags, word2idx, tag2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)


class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_tags, pad_word_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_word_idx)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0.0,
        )
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(hidden_dim * 2, num_tags)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        out = self.dropout(out)
        logits = self.classifier(out)
        return logits

EMB_DIM = 128
HIDDEN_DIM = 128
NUM_TAGS = len(tag2idx)

model = BiLSTMTagger(
    vocab_size=len(word2idx),
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    num_tags=NUM_TAGS,
    pad_word_idx=word2idx[PAD_TOKEN],
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=tag2idx[PAD_TAG])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


def token_accuracy(y_true, y_pred, pad_idx):
    mask = y_true != pad_idx
    correct = (y_true == y_pred) & mask
    total = mask.sum().item()
    if total == 0:
        return 0.0
    return correct.sum().item() / total


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    count = 0

    with torch.no_grad():
        for x, y, _ in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)
            loss = criterion(logits.view(-1, NUM_TAGS), y.view(-1))

            preds = torch.argmax(logits, dim=-1)
            acc = token_accuracy(y, preds, tag2idx[PAD_TAG])

            total_loss += loss.item()
            total_acc += acc
            count += 1

    return total_loss / max(count, 1), total_acc / max(count, 1)



EPOCHS = 5

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_acc_sum = 0.0
    steps = 0

    for x, y, _ in train_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, NUM_TAGS), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        preds = torch.argmax(logits, dim=-1)
        acc = token_accuracy(y, preds, tag2idx[PAD_TAG])

        train_loss_sum += loss.item()
        train_acc_sum += acc
        steps += 1

    train_loss = train_loss_sum / max(steps, 1)
    train_acc = train_acc_sum / max(steps, 1)
    dev_loss, dev_acc = evaluate(model, dev_loader)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Dev Loss: {dev_loss:.4f}, Dev Acc: {dev_acc:.4f}"
    )

test_loss, test_acc = evaluate(model, test_loader)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Token Accuracy: {test_acc:.4f}")

model.eval()
with torch.no_grad():
    sample_x, sample_y, lengths = next(iter(test_loader))
    sample_x = sample_x.to(DEVICE)

    sample_logits = model(sample_x)
    sample_pred = torch.argmax(sample_logits, dim=-1).cpu()

sample_words = test_sents[0]
sample_gold = test_tags[0]
sample_pred_tags = [idx2tag[idx.item()] for idx in sample_pred[0, : len(sample_words)]]

print("\nSample sentence tokens:")
print(sample_words)
print("\nGold POS tags:")
print(sample_gold)
print("\nPredicted POS tags:")
print(sample_pred_tags)

# Simple per-tag support statistics from training set
tag_counter = Counter(tag for seq in train_tags for tag in seq)
print("\nTraining tag distribution (top 10):")
print(tag_counter.most_common(10))

Using device: cpu
Train sentences: 12544
Dev sentences:   2001
Test sentences:  2077
Vocab size: 16656
Number of POS tags: 17
Tags: ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']
Epoch 1/5 | Train Loss: 0.8816, Train Acc: 0.7376 | Dev Loss: 0.5265, Dev Acc: 0.8285
Epoch 2/5 | Train Loss: 0.3882, Train Acc: 0.8772 | Dev Loss: 0.4188, Dev Acc: 0.8667
Epoch 3/5 | Train Loss: 0.2659, Train Acc: 0.9169 | Dev Loss: 0.3558, Dev Acc: 0.8862
Epoch 4/5 | Train Loss: 0.1911, Train Acc: 0.9411 | Dev Loss: 0.3493, Dev Acc: 0.8940
Epoch 5/5 | Train Loss: 0.1397, Train Acc: 0.9574 | Dev Loss: 0.3494, Dev Acc: 0.8974

Test Loss: 0.3501
Test Token Accuracy: 0.8955

Sample sentence tokens:
['What', 'if', 'Google', 'Morphed', 'Into', 'GoogleOS', '?']

Gold POS tags:
['PRON', 'SCONJ', 'PROPN', 'VERB', 'ADP', 'PROPN', 'PUNCT']

Predicted POS tags:
['PRON', 'SCONJ', 'PROPN', 'NOUN', 'ADP', 'NOUN', 'PUNCT']

Training tag dist

In [3]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Collect all predictions and gold labels from test set
all_preds = []
all_golds = []

model.eval()
with torch.no_grad():
    for x, y, _ in test_loader:
        x = x.to(DEVICE)
        logits = model(x)
        preds = torch.argmax(logits, dim=-1).cpu()

        for i in range(len(x)):
            seq_len = (x[i] != word2idx[PAD_TOKEN]).sum().item()
            gold_seq = y[i, :seq_len]
            pred_seq = preds[i, :seq_len]

            all_golds.extend(gold_seq.tolist())
            all_preds.extend(pred_seq.tolist())

# Filter out padding tags
valid_mask = [g != tag2idx[PAD_TAG] for g in all_golds]
all_golds_valid = [g for g, m in zip(all_golds, valid_mask) if m]
all_preds_valid = [p for p, m in zip(all_preds, valid_mask) if m]

# Generate classification report
tag_names = [idx2tag[i] for i in range(len(idx2tag)) if i != tag2idx[PAD_TAG]]
report = classification_report(
    all_golds_valid,
    all_preds_valid,
    labels=[tag2idx[t] for t in tag_names],
    target_names=tag_names,
    digits=4
)

print("=" * 80)
print("DETAILED PER-TAG EVALUATION ON TEST SET")
print("=" * 80)
print(report)

# Confusion matrix summary (most confused tags)
cm = confusion_matrix(all_golds_valid, all_preds_valid)
print("\n" + "=" * 80)
print("TOP MISCLASSIFICATIONS (Tag pairs where model makes most errors)")
print("=" * 80)

errors = []
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if i != j and cm[i, j] > 0:
            errors.append((cm[i, j], idx2tag[i], idx2tag[j]))

errors.sort(reverse=True)
for count, gold, pred in errors[:10]:
    print(f"{count:4d} misclassifications: GOLD={gold:8s} -> PRED={pred:8s}")

DETAILED PER-TAG EVALUATION ON TEST SET
              precision    recall  f1-score   support

         ADJ     0.9259    0.8171    0.8681      1788
         ADP     0.9335    0.9699    0.9513      2025
         ADV     0.8302    0.8908    0.8595      1191
         AUX     0.9850    0.9812    0.9831      1543
       CCONJ     0.9959    0.9905    0.9932       736
         DET     0.9852    0.9852    0.9852      1897
        INTJ     0.9780    0.7355    0.8396       121
        NOUN     0.7260    0.9311    0.8159      4123
         NUM     0.8784    0.6531    0.7492       542
        PART     0.9657    0.9553    0.9605       649
        PRON     0.9754    0.9889    0.9821      2164
       PROPN     0.8050    0.5113    0.6254      2075
       PUNCT     0.9928    0.9858    0.9893      3096
       SCONJ     0.9396    0.8099    0.8699       384
         SYM     0.8148    0.7788    0.7964       113
        VERB     0.9225    0.8914    0.9067      2605
           X     0.5000    0.0238    0.04